# WeldDataWorkbench: indexed dataset overview

This notebook reads the generated SQLite index. It does not scan raw media directly. Set `WORKSPACE` after running `weldtool init` and `weldtool scan`.


In [ ]:
from pathlib import Path
import pandas as pd

from weld_data_workbench.config import load_config
from weld_data_workbench.index.repository import DatasetRepository

WORKSPACE = Path('../_demo/workspace').resolve()
config = load_config(WORKSPACE)
repo = DatasetRepository(config.index_path, config.dataset_root)


In [ ]:
stats = repo.stats()
stats


In [ ]:
samples = pd.DataFrame(repo.iter_samples(batch_size=1000))
samples.head()


In [ ]:
pd.crosstab(samples['category'].fillna('Unknown'), samples['split'].fillna('Unknown'), margins=True)


In [ ]:
samples.groupby(['split', 'health_status'], dropna=False).size().unstack(fill_value=0)


In [ ]:
# Inspect one complete sample contract.
sample_id = samples.iloc[0]['sample_id']
detail = repo.get_sample(sample_id)
detail


## Generate previews for the selected sample


In [ ]:
from weld_data_workbench.previews.generator import PreviewGenerator
bundle = PreviewGenerator(config, repo).generate(sample_id)
bundle


## Load extracted features

Run `weldtool features --workspace <workspace>` first.


In [ ]:
feature_path = config.features_dir / 'features.csv'
features = pd.read_csv(feature_path) if feature_path.exists() else pd.DataFrame()
features.head()
